# TF-IDF Features + LASSO Selection → OLS Inference

## What it does
Builds predictive text signals from **raw article-level text** rather than pre-computed
topic proportions, through a three-stage pipeline:

1. **CountVectorizer** — tokenizes text and counts word (or n-gram) occurrences per period
2. **TfidfTransformer** — re-weights raw counts by term frequency × inverse document frequency,
   downweighting words that appear in every document (stop-word-like) and upweighting rare, informative terms
3. **LASSO path** — scans the regularization path to find the alpha that selects exactly `N_SELECT`
   non-zero TF-IDF features, then refits OLS on those features for interpretable coefficients and inference

## TF-IDF weighting
$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \log\frac{1 + N}{1 + \text{df}(t)}$$

where `tf(t,d)` = frequency of term `t` in document `d`, `N` = total documents, `df(t)` = documents containing `t`.

## When to use it vs `text_lasso.ipynb`
| Notebook | Input | Best for |
|---|---|---|
| `tfidf_lasso.ipynb` | Raw text corpus | Starting from scratch with raw documents |
| `text_lasso.ipynb` | Pre-computed features (topics, embeddings) | Ready-made signal matrix |

## Data format
**Text file** — CSV with at least two columns: a date column and a text column (one row per article or one row per period with concatenated text).
**Outcome file** — CSV with a date column and one or more numeric outcome columns.
Both files are merged on date (aligned to month-start).

## Configuration

In [ ]:
CONFIG = {
    # --- Text corpus file ---
    'TEXT_FILE':        '../../data/articles.csv',   # CSV with text + date columns
    'TEXT_DATE_COL':    'date',
    'TEXT_COL':         'text',                      # column containing raw article text
    # --- Outcome file ---
    'OUTCOME_FILE':     '../../data/macro.csv',
    'OUTCOME_DATE_COL': 'date',
    # --- Targets to predict ---
    'TARGET_COLS':      ['mret'],
    # --- CountVectorizer settings ---
    'MAX_FEATURES':     5000,     # vocabulary size (top-N most frequent terms)
    'NGRAM_RANGE':      [1, 2],   # unigrams and bigrams
    'MIN_DF':           5,        # ignore terms appearing in fewer than N documents
    'MAX_DF':           0.95,     # ignore terms appearing in more than 95% of documents
    'SUBLINEAR_TF':     True,     # apply log(1+tf) instead of raw tf
    # --- LASSO feature selection ---
    'N_SELECT':         10,       # number of TF-IDF features to select
    'N_ALPHAS':         2000,     # resolution of LASSO path
    # --- Output ---
    'SAVE_RESULTS':     True,
    'OUTPUT_DIR':       '../../results',
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Step 1 — Load Text Corpus & Aggregate to Monthly

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('../../').resolve()))
from utils import load_csv

text_df = load_csv(CONFIG['TEXT_FILE'], parse_dates=[CONFIG['TEXT_DATE_COL']])
text_df[CONFIG['TEXT_DATE_COL']] = text_df[CONFIG['TEXT_DATE_COL']].dt.to_period('M').dt.to_timestamp()

print(f'Text corpus : {text_df.shape[0]:,} rows')
print(f'Period      : {text_df[CONFIG["TEXT_DATE_COL"]].min().date()} — {text_df[CONFIG["TEXT_DATE_COL"]].max().date()}')
print(f'Sample text : {str(text_df[CONFIG["TEXT_COL"]].iloc[0])[:120]}...')

In [ ]:
# Aggregate: concatenate all article texts within each month
monthly_text = (
    text_df.groupby(CONFIG['TEXT_DATE_COL'])[CONFIG['TEXT_COL']]
    .apply(lambda texts: ' '.join(texts.dropna().astype(str)))
    .reset_index()
    .rename(columns={CONFIG['TEXT_DATE_COL']: 'date'})
)

print(f'Monthly periods : {len(monthly_text)}')
print(f'Avg words/month : {monthly_text[CONFIG["TEXT_COL"]].str.split().str.len().mean():.0f}')
monthly_text.head()

## Step 2 — Build TF-IDF Feature Matrix

In [ ]:
# Fit CountVectorizer on the full corpus
vectorizer = CountVectorizer(
    max_features  = CONFIG['MAX_FEATURES'],
    ngram_range   = tuple(CONFIG['NGRAM_RANGE']),
    min_df        = CONFIG['MIN_DF'],
    max_df        = CONFIG['MAX_DF'],
    stop_words    = 'english',
)
count_matrix = vectorizer.fit_transform(monthly_text[CONFIG['TEXT_COL']])
vocab        = vectorizer.get_feature_names_out()

# Apply TF-IDF transformation
tfidf_transformer = TfidfTransformer(use_idf=True, smooth_idf=True,
                                     sublinear_tf=CONFIG['SUBLINEAR_TF'])
tfidf_matrix = tfidf_transformer.fit_transform(count_matrix)

# Convert to dense DataFrame (vocabulary may be large; consider keeping sparse for big corpora)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index   = monthly_text['date'],
    columns = vocab,
)

print(f'Vocabulary size : {len(vocab)}')
print(f'TF-IDF matrix   : {tfidf_df.shape}')
print(f'Top 10 terms by mean TF-IDF weight:')
print(tfidf_df.mean().nlargest(10).to_string())

## Step 3 — Load Outcomes & Merge

In [ ]:
outcome_df = load_csv(CONFIG['OUTCOME_FILE'], parse_dates=[CONFIG['OUTCOME_DATE_COL']])
outcome_df[CONFIG['OUTCOME_DATE_COL']] = outcome_df[CONFIG['OUTCOME_DATE_COL']].dt.to_period('M').dt.to_timestamp()

targets = [CONFIG['TARGET_COLS']] if isinstance(CONFIG['TARGET_COLS'], str) else CONFIG['TARGET_COLS']

df = tfidf_df.reset_index().rename(columns={'date': 'date'}).merge(
    outcome_df[['date'] + targets].rename(columns={CONFIG['OUTCOME_DATE_COL']: 'date'})
    if CONFIG['OUTCOME_DATE_COL'] != 'date'
    else outcome_df[['date'] + targets],
    on='date', how='inner',
)

print(f'Merged sample : {len(df)} months')
print(f'Features      : {len(vocab)} TF-IDF terms')
print(f'Targets       : {targets}')
df.head()

## Step 4 — LASSO Path: Select Top k TF-IDF Features per Target

Same two-stage procedure as `text_lasso.ipynb` but applied to TF-IDF features instead of topic proportions.

In [ ]:
def lasso_select_and_ols(X_scaled, y, feature_names, n_select, n_alphas):
    """LASSO path → pick alpha with exactly n_select nonzero → refit OLS."""
    valid = ~np.isnan(y)
    y_v, X_v = y[valid], X_scaled[valid]

    alphas_out, coefs_out, _ = lasso_path(X_v, y_v, n_alphas=n_alphas)
    n_nonzero = np.sum(coefs_out != 0, axis=0)
    idx = np.where(n_nonzero == n_select)[0]
    if len(idx) == 0:
        return None

    pick  = idx[0]
    coefs = coefs_out[:, pick]
    nz    = np.where(coefs != 0)[0]
    sel   = [feature_names[j] for j in nz]

    X_ols = sm.add_constant(pd.DataFrame(X_v[:, nz], columns=sel))
    ols   = sm.OLS(y_v, X_ols).fit()

    return {
        'alpha':       alphas_out[pick],
        'selected':    sel,
        'lasso_coefs': {s: coefs[j] for s, j in zip(sel, nz)},
        'ols_model':   ols,
        'n_obs':       int(valid.sum()),
    }

print('Helper defined.')

In [ ]:
X_feat   = df[vocab].values.astype(float)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_feat)

all_results = {}

for target in targets:
    print(f'\n{"="*60}')
    print(f'TARGET: {target}')
    print('='*60)

    y = df[target].values
    res = lasso_select_and_ols(X_scaled, y, list(vocab),
                               n_select=CONFIG['N_SELECT'],
                               n_alphas=CONFIG['N_ALPHAS'])

    if res is None:
        print(f'  No alpha yielding exactly {CONFIG["N_SELECT"]} nonzero features — try adjusting N_SELECT.')
        continue

    all_results[target] = res
    ols = res['ols_model']

    print(f'  Alpha            : {res["alpha"]:.6f}')
    print(f'  N observations   : {res["n_obs"]}')
    print(f'  Selected features: {res["selected"]}')
    print()
    print(f'  LASSO coefs (standardized):')
    for feat, coef in res['lasso_coefs'].items():
        print(f'    {feat}: {coef:+.6f}')
    print()
    print(f'  OLS R²     : {ols.rsquared:.4f}')
    print(f'  OLS Adj R² : {ols.rsquared_adj:.4f}')
    print()
    print(ols.summary())

## Step 5 — Coefficient Plot per Target

In [ ]:
for target, res in all_results.items():
    ols   = res['ols_model']
    coefs = ols.params.drop('const')
    conf  = ols.conf_int().drop('const')
    pvals = ols.pvalues.drop('const')

    fig, ax = plt.subplots(figsize=(9, max(4, len(coefs) * 0.5)))
    y_pos  = range(len(coefs))
    colors = ['steelblue' if v > 0 else 'tomato' for v in coefs]
    ax.barh(y_pos, coefs.values, color=colors, alpha=0.8)
    ax.errorbar(coefs.values, y_pos,
                xerr=[coefs.values - conf[0].values, conf[1].values - coefs.values],
                fmt='none', color='black', capsize=4, linewidth=1.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'{n} (p={p:.3f})' for n, p in zip(coefs.index, pvals)], fontsize=9)
    ax.invert_yaxis()
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set(xlabel='OLS Coefficient (± 95% CI)',
           title=f'Selected TF-IDF Terms → {target}  (R²={ols.rsquared:.3f})')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## Step 6 — Save Results

In [ ]:
if CONFIG['SAVE_RESULTS']:
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

    rows = []
    for target, res in all_results.items():
        ols = res['ols_model']
        rows.append({
            'target':           target,
            'alpha':            res['alpha'],
            'n_selected':       len(res['selected']),
            'selected_terms':   ', '.join(res['selected']),
            'r2':               ols.rsquared,
            'adj_r2':           ols.rsquared_adj,
            'n_obs':            res['n_obs'],
            'vocab_size':       len(vocab),
            'max_features':     CONFIG['MAX_FEATURES'],
            'ngram_range':      str(CONFIG['NGRAM_RANGE']),
            'sublinear_tf':     CONFIG['SUBLINEAR_TF'],
            'notebook':         'tfidf_lasso',
        })

    path = os.path.join(CONFIG['OUTPUT_DIR'], 'tfidf_lasso_summary.csv')
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f'Saved: {path}')
else:
    print('SAVE_RESULTS = False — skipping.')